# 🤖 Simple RAG Demo — HR Policy Assistant

**RAG** stands for **Retrieval-Augmented Generation**. In plain English:

1. We take a document (here, an HR policy handbook).
2. We chop it into small pieces and turn each piece into a list of numbers (an "embedding") that captures its meaning.
3. We store those pieces in a searchable database (a "vector store").
4. When a user asks a question, we **search** the database for the most relevant pieces, and hand them to an AI model to **generate** a final answer.

That's it. No magic — just search + a language model.

**What we use in this notebook:**
- 📄 Data: `data/hr_policy.txt` (a sample HR policy document)
- 🔢 Embeddings: **Jina AI**
- 🗄️ Vector store: **FAISS**
- 🧠 LLM: **Groq** (fast + free-tier friendly)
- 🕸️ Framework: **LangChain** (`create_agent`)

> Before running: make sure the notebook's kernel is set to the `ragenv` virtual environment (top-right corner of VS Code / Jupyter), and that a `.env` file with `GROQ_API_KEY` and `JINA_API_KEY` exists in this folder.

In [1]:
import os 
from dotenv import load_dotenv


In [2]:
import os 
from dotenv import load_dotenv
# langchain
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_community.embeddings import JinaEmbeddings



C:\Users\Rishi Bije\AppData\Local\Temp\ipykernel_38052\136230262.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
load_dotenv()

True

In [4]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR LOADED")


ENV VAR LOADED


LOADING OUR DATA

In [5]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

#### DATA INGESTION 

In [6]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")

loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()

print(f"Loaded file: {DATA_FILE_PATH}")
print(f"Number of documents loaded: {len(documents)}")
print(f"Total characters in document: {len(documents[0].page_content)}")
print("\n--- Preview of first 300 characters ---")
print(documents[0].page_content[:300])

Loaded file: data\hr_policy.txt
Number of documents loaded: 1
Total characters in document: 2598

--- Preview of first 300 characters ---
COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carr


#### LANGCHAIN DOCUMENT 

Langchain processes everything in form of documents 


DOCUMENTS : 

PAGE CONTENT -- the actual data 

METADATA  - extra information about the data 

In [7]:
len(documents)

1

In [8]:
print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [9]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [10]:
print(f"Total characters in document: {len(documents[0].page_content)}")

Total characters in document: 2598


SPLITTING OUR DATA

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [12]:
len(chunks)

9

In [13]:
from langchain_community.embeddings import JinaEmbeddings

embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("EMB MODEL READY THE NAME IS ", embeddings_model.model_name)


EMB MODEL READY THE NAME IS  jina-embeddings-v2-base-en


In [15]:
from langchain_community.vectorstores import FAISS 

%pip install -U faiss-cpu

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

     ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
     --------------------------------------- 0.0/16.3 MB 960.0 kB/s eta 0:00:17
     ---------------------------------------- 0.1/16.3 MB 1.4 MB/s eta 0:00:12
      --------------------------------------- 0.2/16.3 MB 1.8 MB/s eta 0:00:09
      --------------------------------------- 0.4/16.3 MB 2.5 MB/s eta 0:00:07
      --------------------------------------- 0.4/16.3 MB 2.5 MB/s eta 0:00:07
      --------------------------------------- 0.4/16.3 MB 2.5 MB/s eta 0:00:07
     - -------------------------------------- 0.5/16.3 MB 1.9 MB/s eta 0:00:09
     - -------------------------------------- 0.7/16.3 MB 2.3 MB/s eta 0:00:07
     - -------------------------------------- 0.8/16.3 MB 2.4 MB/s eta 0:00:07
     - -------------------------------------- 0.8/16.3 MB 2.4 MB/s eta 0:00:07
     - -------------------------------------- 0.8/16.3 MB 1.9 MB/s eta 0:00:09
     -- ------------------------------------- 1.0/16.3 MB 


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


NameError: name 'vector_store' is not defined

In [16]:
from langchain_community.vectorstores import FAISS 

vector_store = FAISS.from_documents(chunks , embeddings_model)

print("CHUNKS ARE STORED" , vector_store.index.ntotal)

CHUNKS ARE STORED 9


In [17]:
test_query = "How many sick leaves employees get"

## SIMILARITY SEARCH 

top_matches = vector_store.similarity_search(test_query , k=2)
print(f"Query: {test_query}\n")
for i,match in enumerate(top_matches,start=1):
    print(f"--- Match {i} ---")
    print(match.page_content)
    print()

Query: How many sick leaves employees get

--- Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

--- Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [18]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})  # returns top 3 relevant chunks

def search_hr_policy(question:str)->str:
    """
    Search the HR policy document for information about leave, work from home,
    probation, notice period, reimbursement, code of conduct, holidays, or exit process.
    
    """
    matching_chunks = retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

DATA RETRIVAL
LLM

In [19]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature=0.5  # creativity 
)

llm.model_name

'openai/gpt-oss-120b'

In [20]:
tr = llm.invoke("Hey what is the leave policy")

In [21]:
tr.content

'Sure thing! Could you let me know which company or organization’s leave policy you’re interested in? That way I can give you the most relevant details (e.g., annual/vacation leave, sick leave, parental leave, etc.).'

AI AGENT

3 --

LLM - BRAIN

TOOL - SUPER POWER

MEMORY - no memory

In [22]:
from langchain.agents import create_agent      

In [23]:
hr_assistant = create_agent(
    model = llm,
    tools=[search_hr_policy],
    system_prompt= """ 
    
    You are a friendly HR assistant working for Acme Crop. 
    Always use the search_hr_policy tool to look up 
    facts before answering. 
    If the answer isn't in the search results, say you don't know "
    instead of guessing."
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [24]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [25]:
response 

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='b2a5cbc3-bf09-4c8a-9c32-487e3f7fddb2'),
  AIMessage(content='I’m a friendly HR assistant for **Acme\u202fCrop**.', additional_kwargs={'reasoning_content': 'User asks: "tell me which org you work for". As HR assistant for Acme Crop. Should answer that I work for Acme Crop. No need to search policy. It\'s factual about organization. Could answer directly.'}, response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 209, 'total_tokens': 278, 'completion_time': 0.147480722, 'completion_tokens_details': {'reasoning_tokens': 46}, 'prompt_time': 0.012236561, 'prompt_tokens_details': None, 'queue_time': 0.318308751, 'total_time': 0.159717283}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c800245357', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a15536-7399-7f50-b534-6aa395fa4

In [26]:
response["messages"][-2].content

'tell me which org you work for'

In [27]:
response["messages"][-1].content

'I’m a friendly HR assistant for **Acme\u202fCrop**.'